# 2a Clean metadata

In this notebook, we will clean our metadata. Primarily, we will be de-duplicating our data in preparation for __Mash__ filtration & clustering

In [1]:
import pandas as pd

import numpy as np
import pandas as pd

import seaborn as sns
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker

from tqdm.notebook import tqdm

In [2]:
downloaded_species_metadata = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/interim/1b_ncbi_genome_metadata_filtered.csv', index_col=0, dtype={'genome_id': str})

display(
    # downloaded_species_metadata.shape,
    downloaded_species_metadata.head(),
)

(2676, 81)

,genome_name,organism_name,taxon_id,genome_status,strain,culture_collection,type_strain,completion_date,bioproject_accession,biosample_accession,...,assembly_stats_gc_count,assembly_stats_genome_coverage,assembly_stats_atgc_count,organism_infraspecific_names_strain,organism_infraspecific_names_isolate,assembly_stats_total_number_of_chromosomes,wgs_project_accession,annotation_count_gene_total,annotation_count_gene_protein_coding,annotation_count_gene_pseudogene
genome_id,,,,,,,,,,,,,,,,,,,,,
GCA_000006765.1,Pseudomonas aeruginosa PAO1,Pseudomonas aeruginosa PAO1,208964,Complete,PAO1,NaN,NaN,2006-07-07,PRJNA331,SAMN02603714,...,4169320,NaN,6264404,PAO1,NaN,1.0,NaN,5682.0,5571.0,5.0
GCA_000014625.1,Pseudomonas aeruginosa UCBPP-PA14,Pseudomonas aeruginosa UCBPP-PA14,208963,Complete,UCBPP-PA14,NaN,NaN,2006-10-06,PRJNA386,SAMN02603591,...,4333951,NaN,6537637,UCBPP-PA14,NaN,1.0,NaN,5977.0,5892.0,NaN
GCA_000026645.1,Pseudomonas aeruginosa LESB58,Pseudomonas aeruginosa LESB58,557722,Complete,LESB58,NaN,NaN,2008-12-24,PRJEA31101,SAMEA1705916,...,4376792,NaN,6601757,LESB58,NaN,1.0,NaN,6067.0,5931.0,34.0
GCA_000226155.1,Pseudomonas aeruginosa M18,Pseudomonas aeruginosa M18,941193,Complete,M18,NaN,NaN,2011-09-19,PRJNA61423,SAMN02603849,...,4208227,NaN,6327754,M18,NaN,1.0,NaN,5770.0,5684.0,6.0
GCA_000271365.1,Pseudomonas aeruginosa DK2,Pseudomonas aeruginosa DK2,1093787,Complete,DK2,NaN,NaN,2012-06-20,PRJNA73815,SAMN02603895,...,4243057,NaN,6402658,DK2,NaN,1.0,NaN,5960.0,5884.0,NaN


## De-duplicate entries

In [3]:
########### table contains refseq and genbank versions of genomes -- dropping duplicates so that we keep the refseq genomes in case of duplicates
downloaded_species_metadata = downloaded_species_metadata.sort_values(
    by=["refseq_accessions", "source_database"],
    key=lambda col: col.eq("SOURCE_DATABASE_REFSEQ"),
    ascending=False
)

downloaded_species_metadata = downloaded_species_metadata.drop_duplicates(
    subset="refseq_accessions",
    keep="first"
)

### Ensure `biosample_accession` is unique & drop duplicates

In [4]:
downloaded_species_metadata = downloaded_species_metadata.drop_duplicates(subset=['biosample_accession'])

display(
    downloaded_species_metadata.shape,
    downloaded_species_metadata.head()
)

(1374, 81)

,genome_name,organism_name,taxon_id,genome_status,strain,culture_collection,type_strain,completion_date,bioproject_accession,biosample_accession,...,assembly_stats_gc_count,assembly_stats_genome_coverage,assembly_stats_atgc_count,organism_infraspecific_names_strain,organism_infraspecific_names_isolate,assembly_stats_total_number_of_chromosomes,wgs_project_accession,annotation_count_gene_total,annotation_count_gene_protein_coding,annotation_count_gene_pseudogene
genome_id,,,,,,,,,,,,,,,,,,,,,
GCF_000006765.1,Pseudomonas aeruginosa PAO1,Pseudomonas aeruginosa PAO1,208964,Complete,PAO1,NaN,NaN,2006-07-24,PRJNA331,NaN,...,4169320,NaN,6264404,PAO1,NaN,1.0,NaN,5697.0,5572.0,19.0
GCF_000014625.1,Pseudomonas aeruginosa UCBPP-PA14,Pseudomonas aeruginosa UCBPP-PA14,208963,Complete,UCBPP-PA14,NaN,NaN,2006-10-06,PRJNA386,SAMN02603591,...,4333951,NaN,6537637,UCBPP-PA14,NaN,1.0,NaN,6041.0,5891.0,70.0
GCF_000026645.1,Pseudomonas aeruginosa LESB58,Pseudomonas aeruginosa LESB58,557722,Complete,LESB58,NaN,NaN,2008-12-24,PRJEA31101,SAMEA1705916,...,4376792,NaN,6601757,LESB58,NaN,1.0,NaN,6195.0,6059.0,52.0
GCF_000168335.1,Pseudomonas aeruginosa PACS2,Pseudomonas aeruginosa PACS2,388272,Complete,PACS2,NaN,NaN,2006-06-05,PRJNA16851,SAMN02471994,...,4306352,NaN,6492423,PACS2,NaN,1.0,AAQW01,6043.0,5936.0,28.0
GCF_000223945.1,Pseudomonas aeruginosa 19BR,Pseudomonas aeruginosa 19BR,1051003,Complete,19BR,NaN,NaN,2011-08-17,PRJNA70773,SAMN02471401,...,4458874,33.0,6742964,19BR,NaN,1.0,AFXJ01,6332.0,6163.0,88.0


### (Optional) Ensure assembly accession (i.e., index) is unique
All unique so not dropping anything

In [5]:
downloaded_species_metadata.reset_index().value_counts('genome_id')

genome_id
GCA_048593205.1    1
GCF_041154335.1    1
GCF_041519535.1    1
GCF_041519425.1    1
GCF_041514425.1    1
                  ..
GCF_027570455.1    1
GCF_027570435.1    1
GCF_027570415.1    1
GCF_027359235.1    1
GCF_976988945.1    1
Name: count, Length: 1374, dtype: int64

In [6]:
output_path = '/mnt/craig/pan_phylon/P_aeruginosa/data/interim/2a_ncbi_genome_metadata_filtered.csv'
downloaded_species_metadata.reset_index().to_csv(output_path, index=False)
output_path

'/mnt/craig/pan_phylon/P_aeruginosa/data/interim/2a_ncbi_genome_metadata_filtered.csv'

In [21]:
downloaded_species_metadata.loc['GCA_048593205.1']

genome_name                                   Pseudomonas aeruginosa ps1
organism_name                                     Pseudomonas aeruginosa
taxon_id                                                             287
genome_status                                                   Complete
strain                                                               ps1
                                                         ...            
assembly_stats_total_number_of_chromosomes                           1.0
wgs_project_accession                                                NaN
annotation_count_gene_total                                       5601.0
annotation_count_gene_protein_coding                              5416.0
annotation_count_gene_pseudogene                                   121.0
Name: GCA_048593205.1, Length: 81, dtype: object

In [7]:
downloaded_species_metadata.genome_status.value_counts()

genome_status
Complete    1374
Name: count, dtype: int64

In [8]:
# all downloaded files
downloaded = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/interim/1b_ncbi_genome_metadata_filtered.csv', index_col=0, dtype={'genome_id': str})
len(downloaded.index.values)

2676

In [9]:
# count of shared files (should be equal to number of strains in final downloaded metadata)
len(set(downloaded.index.values).intersection(set(downloaded_species_metadata.index.values)))

1374

In [10]:
# need to remove from downloaded files
remove = set(downloaded.index.values).difference(set(downloaded_species_metadata.index.values))
len(remove)

1302